In [ ]:
from langchain_anyllm import ChatAnyLLM

llm = ChatAnyLLM(
        # model="qwen/qwen3.5-9b",   # must match the model loaded in LMStudio
        # model="qwen3.5-4b",   # must match the model loaded in LMStudio
        provider='lmstudio',
        model="qwen3.5-27b-mtp",
        api_base="http://localhost:1234/v1",
        api_key="not-needed",
        # model_kwargs={
        #     "tool_choice": "auto",
        # },
    )

In [9]:
llm.profile

In [24]:
# llm.invoke('who are you')

In [25]:
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_community.tools import DuckDuckGoSearchRun
tool = DuckDuckGoSearchRun()

In [26]:
from typing import Literal
from urllib.parse import urlparse

import httpx
import trafilatura
from bs4 import BeautifulSoup
from ddgs import DDGS
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

from deepagents import create_deep_agent


class WebSearchInput(BaseModel):
    query: str = Field(..., description="Search query")
    max_results: int = Field(default=5, ge=1, le=10)
    region: str = Field(default="sg-en")
    safesearch: Literal["on", "moderate", "off"] = Field(default="moderate")
    timelimit: Literal["d", "w", "m", "y"] | None = Field(default=None)


@tool("web_search", args_schema=WebSearchInput)
def web_search(
    query: str,
    max_results: int = 5,
    region: str = "sg-en",
    safesearch: str = "moderate",
    timelimit: str | None = None,
) -> str:
    """Search the web using DDGS. Returns titles, snippets, and URLs."""
    try:
        with DDGS() as ddgs:
            results = list(
                ddgs.text(
                    query,
                    region=region,
                    safesearch=safesearch,
                    timelimit=timelimit,
                    max_results=max_results,
                )
            )

        if not results:
            return "No search results found."

        lines = []
        for i, r in enumerate(results, start=1):
            title = r.get("title", "").strip()
            href = r.get("href", "").strip()
            body = r.get("body", "").strip()

            lines.append(
                f"{i}. {title}\n"
                f"URL: {href}\n"
                f"Snippet: {body}"
            )

        return "\n\n".join(lines)

    except Exception as e:
        return f"Search failed: {type(e).__name__}: {e}"


class ScrapeWebpageInput(BaseModel):
    url: str = Field(..., description="The URL to scrape")
    max_chars: int = Field(default=6000, ge=1000, le=20000)


@tool("scrape_webpage", args_schema=ScrapeWebpageInput)
def scrape_webpage(url: str, max_chars: int = 6000) -> str:
    """Scrape readable text from a webpage URL."""
    try:
        parsed = urlparse(url)

        if parsed.scheme not in {"http", "https"}:
            return "Invalid URL: only http and https URLs are allowed."

        headers = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0 Safari/537.36"
            )
        }

        with httpx.Client(
            headers=headers,
            follow_redirects=True,
            timeout=20,
        ) as client:
            response = client.get(url)
            response.raise_for_status()

        html = response.text

        extracted = trafilatura.extract(
            html,
            include_comments=False,
            include_tables=True,
            output_format="txt",
        )

        if not extracted:
            soup = BeautifulSoup(html, "html.parser")

            for tag in soup(["script", "style", "noscript", "svg"]):
                tag.decompose()

            extracted = soup.get_text(separator="\n", strip=True)

        if not extracted:
            return f"Could not extract readable text from {url}"

        extracted = "\n".join(
            line.strip()
            for line in extracted.splitlines()
            if line.strip()
        )

        if len(extracted) > max_chars:
            extracted = extracted[:max_chars] + "\n\n[Truncated]"

        return f"URL: {url}\n\n{extracted}"

    except httpx.HTTPStatusError as e:
        return f"HTTP error while scraping {url}: {e.response.status_code}"
    except httpx.RequestError as e:
        return f"Request error while scraping {url}: {type(e).__name__}: {e}"
    except Exception as e:
        return f"Scrape failed for {url}: {type(e).__name__}: {e}"



In [43]:
instructions = """
You are a web research agent.

You have two web tools:
- web_search: finds URLs.
- scrape_webpage: reads one URL.

Always follow this order:
1. Call web_search.
2. Choose the best URLs.
3. Call scrape_webpage on those URLs.
4. Answer using the scraped page text.

Never answer from web_search snippets alone unless every scrape_webpage call fails.
"""

agent = create_deep_agent(
    tools=[
        web_search,
        scrape_webpage,
    ],
    system_prompt=instructions,
    model=llm,
)

In [46]:
def run_streaming_research(question: str):
    input_data = {
        "messages": [
            {
                "role": "user",
                "content": question,
            }
        ]
    }

    final_messages = None

    for item in agent.stream(
        input_data,
        stream_mode=["updates", "messages"],
    ):
        if isinstance(item, tuple) and len(item) == 2:
            mode, chunk = item

            if mode == "updates":
                print("\n\n--- update ---")
                print(chunk)

                # Capture latest messages from update chunks
                for node_update in chunk.values():
                    if isinstance(node_update, dict) and "messages" in node_update:
                        final_messages = node_update["messages"]

            elif mode == "messages":
                token, metadata = chunk
                if token.content:
                    print(token.content, end="", flush=True)

        elif isinstance(item, dict):
            mode = item.get("type")
            data = item.get("data")

            if mode == "updates":
                print("\n\n--- update ---")
                print(data)

                for node_update in data.values():
                    if isinstance(node_update, dict) and "messages" in node_update:
                        final_messages = node_update["messages"]

            elif mode == "messages":
                token, metadata = data
                if token.content:
                    print(token.content, end="", flush=True)

    if not final_messages:
        return None

    # Last message is usually the final assistant message
    return final_messages[-1]


response = run_streaming_research(
    "check yahoo finance for stock price of space x on jun 18 2026"
)



--- update ---
{'PatchToolCallsMiddleware.before_agent': None}




--- update ---
{'model': {'messages': [AIMessage(content='\n\n', additional_kwargs={'reasoning_content': "The user wants me to check Yahoo Finance for SpaceX's stock price on June 18, 2026. However, there are a few issues here:\n\n1. SpaceX is a private company and doesn't have publicly traded stock on any exchange\n2. The date requested (June 18, 2026) is in the future - we're currently in 2024 or early 2025\n\nLet me search Yahoo Finance to see what information is available about SpaceX's stock status.\n"}, response_metadata={}, id='lc_run--019ed679-1f6e-73e2-8390-29f74e9075c0', tool_calls=[{'name': 'web_search', 'args': {'query': 'SpaceX stock price Yahoo Finance June 18 2026'}, 'id': '265860162', 'type': 'tool_call'}], invalid_tool_calls=[])]}}


--- update ---
{'TodoListMiddleware.after_model': None}
1. SpaceX stock is getting bought by a wave of retail investors
URL: https://sg.finance.yahoo.com/news/spacex-stoc

In [47]:
print(response.content)



Based on the search results, I found information about SpaceX's IPO which occurred in June 2026:

**Key findings:**
- **IPO Date**: June 12, 2026 (SpaceX began trading on Nasdaq under ticker SPCX)
- **IPO Price**: $135 per share
- **First Day Close (June 12)**: $160.95

**Price progression:**
- **June 12, 2026**: Opened at $150, closed at $160.95 (+19.2% from IPO price)
- **June 13, 2026 (Monday)**: Closed at $178 (+11%)
- **June 14, 2026 (Tuesday)**: Premarket trading showed prices above $200 (+10%)

**For June 18, 2026**: This would be a Friday. The article I found is dated "1 day ago" from the current date and discusses Tuesday's premarket activity showing SpaceX opening above $200 for the first time. However, I couldn't access the specific historical price data page on Yahoo Finance (it returned a 404 error).

Based on the trend described:
- June 12: $160.95 close
- June 13: $178 close  
- June 14: Above $200 in premarket

The stock was on an upward trajectory during that week, b

In [4]:
from pprint import pprint

def inspect_model(model):
    print("\n=== MODEL OBJECT ===")
    print(type(model))

    print("\n=== DIRECT ATTRIBUTES ===")
    for name in [
        "model",
        "model_name",
        "provider",
        "profile",
        "max_tokens",
        "model_kwargs",
    ]:
        print(f"{name}: {getattr(model, name, None)}")

    profile = getattr(model, "profile", None)

    print("\n=== PROFILE ===")
    pprint(profile)

    print("\n=== max_input_tokens ===")
    if isinstance(profile, dict):
        print(profile.get("max_input_tokens"))
    else:
        print(None)

    print("\n=== DIR CONTAINS PROFILE-ish FIELDS ===")
    for name in dir(model):
        lowered = name.lower()
        if "profile" in lowered or "token" in lowered or "context" in lowered:
            print(name)

In [5]:
inspect_model(llm)


=== MODEL OBJECT ===
<class 'langchain_anyllm.chat_models.ChatAnyLLM'>

=== DIRECT ATTRIBUTES ===
model: google/gemma-4-e4b
model_name: None
provider: lmstudio
profile: None
max_tokens: None
model_kwargs: {}

=== PROFILE ===
None

=== max_input_tokens ===
None

=== DIR CONTAINS PROFILE-ish FIELDS ===
_check_profile_keys
_resolve_model_profile
_set_model_profile
custom_get_token_ids
get_num_tokens
get_num_tokens_from_messages
get_token_ids
max_tokens
profile


In [ ]:
# from pprint import pprint
# from langchain.chat_models import init_chat_model


# def inspect_model(model):
#     print("\n=== MODEL OBJECT ===")
#     print(type(model))

#     profile = getattr(model, "profile", None)

#     print("\n=== PROFILE ===")
#     pprint(profile)

#     print("\n=== max_input_tokens ===")
#     if isinstance(profile, dict):
#         print(profile.get("max_input_tokens"))
#     else:
#         print(None)


# def main():
#     model = init_chat_model(
#         "openai:gpt-4.1",  # replace with your AnyLLM-style init string if supported
#         temperature=0,
#     )

#     inspect_model(model)


# if __name__ == "__main__":
#     main()

In [2]:
from deepagents import create_deep_agent, FilesystemPermission
from deepagents.backends import FilesystemBackend
from langchain_quickjs import CodeInterpreterMiddleware
from langgraph.checkpoint.memory import MemorySaver

root = r'D:\Projects\mira'


# # ---------------------------------------------------------------------------
# # Human-in-the-loop: which tools need approval and what decisions are allowed
# # deepagents pauses execution, we collect decisions, then resume with Command
# # ---------------------------------------------------------------------------
# INTERRUPT_ON = {
#     # Filesystem reads — safe, no approval needed
#     "read_file": False,
#     "ls":        False,
#     "glob":      False,
#     "grep":      False,
#     # Writes — show diff, let user approve/edit/reject
#     "write_file": {"allowed_decisions": ["approve", "edit", "reject"]},
#     "edit_file":  {"allowed_decisions": ["approve", "edit", "reject"]},
#     # eval (interpreter) — approve/reject; no edit since it's a code block
#     "eval":       {"allowed_decisions": ["approve", "reject"]},
# }

# # ---------------------------------------------------------------------------
# # PTC allowlist: which tools the interpreter can call from TypeScript code.
# # e.g. the agent can write:
# #   const files = await tools.glob({ pattern: "**/*.py" });
# #   const contents = await Promise.all(files.map(f => tools.readFile({ path: f })));
# # ---------------------------------------------------------------------------
# PTC_TOOLS = ["read_file", "write_file", "edit_file", "ls", "glob", "grep"]


# # FilesystemBackend provides ls/read_file/write_file/edit_file/glob/grep.
# # virtual_mode=True enforces root_dir-relative path semantics.
backend = FilesystemBackend(root_dir=root, virtual_mode=True)

# # First-match-wins permission rules.
# permissions = [
#     # Never expose secrets
#     FilesystemPermission(
#         operations=["read", "write"],
#         paths=["/**/.env", "/**/.env.*", "/**/*.pem", "/**/*.key"],
#         mode="deny",
#     ),
#     # Allow everything else inside the workspace
#     FilesystemPermission(
#         operations=["read", "write"],
#         paths=["/**"],
#         mode="allow",
#     ),
# ]

# # CodeInterpreterMiddleware adds the `eval` tool.
# # ptc= allowlists which agent tools are callable from inside TypeScript
# # as async functions under the global `tools` namespace.
# interpreter = CodeInterpreterMiddleware(ptc=PTC_TOOLS)

# system_prompt = (
#     "You are Mira (Minimal Iterative Reasoning Agent), a helpful coding assistant running in the user's terminal.\n"
#     "You have filesystem tools (read_file, write_file, edit_file, ls, glob, grep) "
#     "and an eval tool that runs TypeScript in a persistent QuickJS interpreter.\n"
#     "Use the interpreter (eval) when you need to: loop over many files, run "
#     "parallel tool calls with Promise.all, filter/aggregate data, or keep "
#     "intermediate state out of the model context.\n"
#     "Always explain what you are about to do before doing it. "
#     "Be concise. Prefer showing code over describing it. "
#     "If a task is ambiguous, ask one clarifying question before proceeding."
# )

# agent = create_deep_agent(
#     model=llm,
#     backend=backend,
#     permissions=permissions,
#     middleware=[interpreter],
#     interrupt_on=INTERRUPT_ON,
#     # checkpointer=MemorySaver(),
#     system_prompt=system_prompt,
# )

In [7]:
agent = create_deep_agent(
    model=llm,
    backend=backend,
    # permissions=permissions,
    # middleware=[interpreter],
    # interrupt_on=INTERRUPT_ON,
    # checkpointer=MemorySaver(),
    # system_prompt=system_prompt,
)

In [8]:
user_input = 'what files are in the root folder'
agent_input = {"messages": [{"role": "user", "content": user_input}]}

In [9]:
# run = agent.stream_events(
#     agent_input, 
#     config={"configurable": {"thread_id": 'xxxxx'}}, 
#     version="v3"
#     )


# for message in run.messages:
#     header_printed = False

#     for delta in message.reasoning:
#         if delta:
#             # print_reasoning_header()
#             # print_reasoning_token(delta)
#             # print(delta)
#             pass

#     for delta in message.text:
#         if delta:
#             if not header_printed:
#                 # print_agent_header()
#                 header_printed = True
#             # print(delta)

#     if header_printed:
#         # print_newline()
#         # print()
#         pass

# # --- Tool calls: finalized name + args + output ---
# # run.tool_calls is a separate projection that does NOT share state with
# # run.messages, so iterating it after messages is safe.
# for call in run.tool_calls:
#     args = call.input if isinstance(call.input, dict) else {"input": call.input}
#     # print_tool_call(call.tool_name, args)
#     print(call.tool_name, args)

#     output_parts = [str(d) for d in call.output_deltas]
#     if output_parts:
#         # print_tool_result(call.tool_name, "".join(output_parts))
#         print(call.tool_name, "".join(output_parts))

#     if call.error:
#         print(f"[{call.tool_name}] {call.error}")

In [18]:
import json

In [19]:
def _parse_tool_call_chunks(message) -> None:
    """Accumulate tool_call_chunks by id, parse complete JSON args, print once."""
    pending: dict[str, dict] = {}
    for chunk in message.tool_calls:
        call_id = chunk.get("id") or str(chunk.get("index", 0))
        if call_id not in pending:
            pending[call_id] = {"name": "", "args_raw": ""}
        if chunk.get("name"):
            pending[call_id]["name"] = chunk["name"]
        if chunk.get("args") is not None:
            pending[call_id]["args_raw"] += chunk["args"]
 
    for call in pending.values():
        if not call["name"]:
            continue
        args: dict = {}
        raw = call["args_raw"].strip()
        if raw:
            try:
                args = json.loads(raw)
            except Exception:
                args = {"args": raw}
        print(call["name"], args)

In [22]:
from langchain.agents import create_agent
from langchain.messages import AIMessageChunk
from langchain_anthropic import ChatAnthropic
from langchain_core.runnables import Runnable

user_input = 'what files are in testing.ipynb'
agent_input = {"messages": [{"role": "user", "content": user_input}]}

agent = create_deep_agent(
    model=llm,
    backend=backend,
)

stream = agent.stream_events(agent_input, version="v3")

for message in stream.messages:

    for delta in message.reasoning:
        print(f"[thinking] {delta}", end="", flush=True)

    for delta in message.text:
        print(delta, end="", flush=True)

    for chunk in message.tool_calls:
        print(f"tool call chunk: {chunk}")

    finalized = message.tool_calls.get()
    if finalized:
        print(f"finalized tool calls: {finalized}")

for call in stream.tool_calls:
    print(f"{call.tool_name}({call.input})")
    for delta in call.output_deltas:
        print(delta, end="", flush=True)
    print(call.output, call.error)

    # _parse_tool_call_chunks(message)

tool call chunk: {'type': 'tool_call_chunk', 'id': '320683432', 'name': 'read_file', 'args': '', 'index': 0}
tool call chunk: {'type': 'tool_call_chunk', 'id': '320683432', 'name': 'read_file', 'args': '{"file_path":"/testing.ipynb"}', 'index': 0}
finalized tool calls: [{'type': 'tool_call', 'id': '320683432', 'name': 'read_file', 'args': {'file_path': '/testing.ipynb'}}]
The `testing.ipynb` file contains **4 code cells**:

1. **Cell 1** (empty) - No content
2. **Cell 2** (empty) - No content  
3. **Cell 3** (empty) - No content
4. **Cell 4** (execution_count: 1) - Contains a `ChatOpenAI` initialization for the Qwen model

The file appears to be mostly empty with only one cell containing actual code that initializes an LLM client.

In [ ]:
stream = agent.stream_events(agent_input, version="v3")

for message in stream.messages:
    for delta in message.reasoning:
        print(f"[thinking] {delta}", end="", flush=True)

    for delta in message.text:
        print(delta, end="", flush=True)





Files in the root folder:
- `.git/` (directory)
- `.gitignore`
- `.mira_sessions/` (directory)
- `mira/` (directory)
- `mira_prompt.md`
- `testing.ipynb`

In [ ]:
stream = agent.stream_events(agent_input, version="v3")

for message in stream.messages:
    print(f"[{message.node}] ", end="")
    for delta in message.text:
        print(delta, end="", flush=True)

    full_message = message.output
    usage = full_message.usage_metadata
    if usage:
        print(usage)

[model] 

[model] 